In [ ]:
# Preparing input variable for clustering temperature zones based on climatology of predictor -> predictand relationship

# V1 - 2026-08-17 - for final run - JPK

# Workflow
# stack -> aggregate -> reproject -> save

In [ ]:
# libraries
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rioxarray as rxr
import xarray as xr
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
import dask
import sys

# Path 
out_path = r"..\Data_Temp\Processed\Spatial_clustering\Climatology_files"
os.makedirs(out_path, exist_ok=True)

sample_raster = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")

chirts_files_path = r"..\Data_Temp\Datasets\CHIRTS"
fetch_rasters_path = r"..\Data\Processed\Fetch"
fetch_rasters_temp_path = r"..\Data\Processed\Fetch\Preprocessing"
dem_rasters_path = r"..\Data\Datasets\DEM_1km"

# Helper - clip and resample raster to ROI at 1km resolution
def clip_and_resample_raster(input_raster):
    resampled_raster = input_raster.rio.reproject_match(sample_raster, resampling= Resampling.bilinear)
    return resampled_raster

In [ ]:
# Create temperature climatology using the finest resolution dataset - CHIRTS

#tmax
chirts_tmax_files = sorted(glob.glob(os.path.join(chirts_files_path, "CHIRTS_tmax_*.tif")))
chirts_tmax_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in chirts_tmax_files]

chirts_tmax_files_rxr = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True) for f in chirts_tmax_files], 
               dim=pd.Index(chirts_tmax_dates, name='time'), join = 'exact')

tmax_climatology = chirts_tmax_files_rxr.resample(time='1YE').mean().mean(dim='time')
tmax_clim = clip_and_resample_raster(tmax_climatology)
tmax_clim.rio.to_raster(os.path.join(out_path_tmax, "tmax_climatology.tif"))

del chirts_tmax_files, chirts_tmax_dates, chirts_tmax_files_rxr, tmax_climatology, tmax_clim  

#tmin 
chirts_tmin_files = sorted(glob.glob(os.path.join(chirts_files_path, "CHIRTS_tmin_*.tif")))
chirts_tmin_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in chirts_tmin_files]

chirts_tmin_files_rxr = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True) for f in chirts_tmin_files], 
               dim=pd.Index(chirts_tmin_dates, name='time'), join = 'exact')

tmin_climatology = chirts_tmin_files_rxr.resample(time='1YE').mean().mean(dim='time')
tmin_clim = clip_and_resample_raster(tmin_climatology)
tmin_clim.rio.to_raster(os.path.join(out_path_tmin, "tmin_climatology.tif"))

del chirts_tmin_files, chirts_tmin_dates, chirts_tmin_files_rxr, tmin_climatology, tmin_clim 

In [ ]:
# Resample and save fetch rasters

fetch_rasters = ["fetch_dir_500m_resampled.tif", "fetch_dist_500m_resampled.tif", 
                 "fetch_dir_1000m_resampled.tif", "fetch_dist_1000m_resampled.tif"]

for fetch_raster in fetch_rasters:
    read_raster = rxr.open_rasterio(os.path.join(fetch_rasters_path, fetch_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(out_path, fetch_raster))
    del read_raster, resampled_raster

#prepare additional (temporary) fetch variables

fetch_temp_rasters = ["local_relief.tif", "dem_smoothed.tif"]

for fetch_temp_raster in fetch_temp_rasters:
    read_raster = rxr.open_rasterio(os.path.join(fetch_rasters_temp_path, fetch_temp_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(out_path, fetch_temp_raster))
    del read_raster, resampled_raster

In [ ]:
# Create GLDAS secondary variables climatology
gldas_variables = ['air_temp', 'lwdown', 'specific_humidity', 'surface_pressure', 'swdown', 'wind_speed']

for variable in gldas_variables:
    var_path = os.path.join(r"..\Data\Datasets\GLDAS", variable)
    var_files = sorted(glob.glob(os.path.join(var_path, "*.tif")))
    var_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in var_files]

    var_files_rxr = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}) for f in var_files], 
               dim=pd.Index(var_dates, name='time'), join = 'exact')
    var_climatology = var_files_rxr.resample(time='1YE').mean().mean(dim='time')
    var_clim = clip_and_resample_raster(var_climatology)
    var_clim.rio.to_raster(os.path.join(out_path, "GLDAS_" + variable + "_climatology.tif"))
    
    del var_files_rxr, var_climatology, var_clim

In [ ]:
# Prepare DEM rasters

dem_rasters = ['DEM_1km_aspect_mean.tif', 'DEM_1km_elevation_max.tif', 
               'DEM_1km_elevation_mean.tif', 'DEM_1km_elevation_min.tif',
               'DEM_1km_elevation_std.tif', 'DEM_1km_slope_mean.tif']

for dem_raster in dem_rasters:
    read_raster = rxr.open_rasterio(os.path.join(dem_rasters_path, dem_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(out_path, dem_raster))
    del read_raster, resampled_raster

In [ ]:
# Prepare MODIS variables
from tqdm.notebook import tqdm 

modis_variables = ['lst_n', 'lst_d', 'evi', 'lst_diff']

for variable in tqdm(modis_variables, desc="Variables"):
    var_path = os.path.join(dataset_dir, "MODIS_2_" + variable)
    var_files = sorted(glob.glob(os.path.join(var_path, "*.tif")))
    var_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in var_files]
    
    df = pd.DataFrame({'file': var_files, 'date': var_dates})
    df['year'] = df['date'].dt.year

    yearly_means = []
    for year, group in tqdm(list(df.groupby('year')), desc=f"Years ({variable})", leave=False):
        resampled_arrays = []
        for f, date in zip(group['file'], group['date']):
            arr = rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True)
            arr_resampled = clip_and_resample_raster(arr)
            arr_resampled = arr_resampled.expand_dims(time=[date])  
            resampled_arrays.append(arr_resampled)
        
        ds = xr.concat(resampled_arrays, dim='time', join = 'exact')
        yearly_mean = ds.mean(dim='time', skipna=True, keep_attrs=True)
        yearly_means.append(yearly_mean)
        del resampled_arrays, ds

    stacked_means = xr.concat(yearly_means, dim='year')

    # Calculate climatological mean
    var_climatology = stacked_means.mean(dim='year', skipna=True)
    output_path = os.path.join(out_path, "MODIS_" + variable + "_climatology.tif")
    var_climatology.rio.to_raster(
        output_path,
        dtype='float32',
        compress='lzw',
        nodata = np.nan  
    )
    
    del df, yearly_means, var_climatology

In [ ]:
#print list of variables prepared
print("Prepared variables for creating temperature clusters:"
      "\n1. Tmax and Tmin Climatology (CHIRTS)"
      "\n2. Fetch variables (dir and dist at 500m and 1000m)"
      "\n3. GLDAS secondary variables climatology - 'air_temp', 'lwdown', 'specific_humidity', 'surface_pressure', 'swdown', 'wind_speed'"
      "\n4. DEM variables (aspect_mean, elevation_max, elevation_mean, elevation_min, elevation_std, slope_mean)"
      "\n5. MODIS variables climatology (evi, lst_diff, lst_n, lst_d)"
      "\n All variables saved to: Data_Temp\Processed\Spatial_clustering\Climatology_files after resampling to 1km grid and reprojecting to WGS84")